In [ ]:
!pip install -q fastapi uvicorn nest-asyncio pycloudflared transformers torch accelerate pydantic

import nest_asyncio, uvicorn, threading, re
from fastapi import FastAPI
from pydantic import BaseModel
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from pycloudflared import try_cloudflare


app = FastAPI()


model_id = "Qwen/Qwen3-1.7B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)


class PromptRequest(BaseModel):
    prompt: list | str
    system_instruction: str | None = None
    max_tokens: int = 1024
    temperature: float = 0.3

@app.post("/generate")
def generate_text(req: PromptRequest):
    print(">>> [Colab] Received request from VS Code")

    # Normaliza messages
    if isinstance(req.prompt, list):
        messages = req.prompt
    else:
        messages = []
        if req.system_instruction:
            messages.append({"role": "system", "content": req.system_instruction})
        messages.append({"role": "user", "content": str(req.prompt)})

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)


    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=1500,
            temperature=0.2,
            do_sample=True
        )

    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    raw_response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    # Strip out the entire <think>...</think> block
    clean_response = re.sub(r"<think>.*?</think>", "", raw_response, flags=re.DOTALL).strip()

    # Handle unclosed </think> tag caused by token truncation
    if "<think>" in clean_response and "</think>" not in clean_response:
        clean_response = clean_response.split("<think>")[-1].strip()

    print(">>> [Colab] Response generation complete! Sending back to local")
    return {"response": clean_response}

nest_asyncio.apply()
public_url = try_cloudflare(port=8000)
print(f"URL For Vscode Local: {public_url.tunnel}")


def start_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)


server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()
print("Server FastAPI run successfully")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (4 outputs left out)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Download cloudflared...:   0%|          | 0/40274445 [00:00<?, ?it/s]

 * Running on https://broad-drama-attend-ability.trycloudflare.com
 * Traffic stats available on http://127.0.0.1:20241/metrics
URL For Vscode Local: https://broad-drama-attend-ability.trycloudflare.com
Server FastAPI run successfully
